# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [36]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, detect_app_version,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    # Frequency Test analysis plots
    plot_ft_depression_curve, plot_ft_averaged_waveforms, plot_ft_peak_curve,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
    make_viewer, compute_h_comparison_data, plot_h_reflex_comparison,
    load_all_recordings, FT_SNAP_HZ, build_settings_dataframe,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [37]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [
    
    # HRPilot-23 Recordings
    #("HRPILOT-23 250US",  "Calibration/HRPilot-23/CALIB1_HRPILOT-23_BOOTH2_250US_10KHZ_8-31-26",        10000.0),
    #("HRPILOT-23 100US",  "Calibration/HRPilot-23/CALIB2_HRPILOT-23_BOOTH3_100US_10KHZ_9-2-26",        10000.0),
    #("FT1 HRPILOT-23 250US",  "Calibration/HRPilot-23/FT1_HRPILOT-23_100US_10KHZ_9-8-26",        10000.0),
    ("Calib3 HRPILOT-23 100US",  "Calibration/HRPilot-23/CALIB3_HRPILOT-23_BOOTH3_100US_10KHZ_9-15-26",        10000.0),
    ("Calib4 HRPILOT-23 100US",  "Calibration/HRPilot-23/CALIB4_HRPILOT-23_BOOTH3_100US_10KHZ_9-15-26",        10000.0),
    
    # HRPilot-25 Recordings
    #("HRPILOT-25 250US",  "Calibration/HRPilot-25/CALIB1_HRPILOT-25_BOOTH2_250US_10KHZ_8-25-26",        10000.0),
    #("HRPILOT-25 100US",  "Calibration/HRPilot-25/CALIB2_HRPILOT-25_BOOTH2_100US_10KHZ_9-2-26",        10000.0),
    #("Calib3 HRPILOT-25 250US",  "Calibration/HRPilot-25/CALIB3_HRPILOT-25_BOOTH1_250US_10KHZ_9-10-26",        10000.0),
    ("Calib4 HRPILOT-25 100US",  "Calibration/HRPilot-25/CALIB4_HRPILOT-25_BOOTH3_100US_10KHZ_9-14-26",        10000.0),
        
    
    # HRPilot-26 Recordings
    #("HRPILOT-26 250US",  "Calibration/HRPilot-26/CALIB1_HRPILOT-26_BOOTH2_250US_10KHZ_8-27-26",        10000.0),
    #("HRPILOT-26 100US",  "Calibration/HRPilot-26/CALIB2_HRPILOT-26_BOOTH1_100US_10KHZ_9-2-26",        10000.0),
    
    # HRPilot-34 Recordings
    #("HRPILOT-34 250US",  "Calibration/HRPilot-34/CALIB1_HRPILOT-34_BOOTH2_250US_10KHZ_9-2-26",        10000.0),
    #("Calib2 HRPILOT-34 250US",  "Calibration/HRPilot-34/CALIB2_HRPILOT-34_BOOTH2_250US_10KHZ_9-10-26",        10000.0),
    ("CM1 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM1_HRPILOT-34_BOOTH1_250US_10KHZ_9-14-26",        10000.0),
    ("CM2 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM2_HRPILOT-34_BOOTH1_250US_10KHZ_9-15-26",        10000.0),
    ("Calib3 HRPILOT-34 100US",  "Calibration/HRPilot-34/CALIB3_HRPILOT-34_BOOTH1_250US_10KHZ_9-15-26",        10000.0),
    ("Calib3.5 HRPILOT-34 100US",  "Calibration/HRPilot-34/CALIB3_HRPILOT-24_BOOTH1_250US_10KHZ_9-15-26",        10000.0),
        
    
    # HRPilot-36 Recordings
    #("HRPILOT-36 250US",  "Calibration/HRPilot-36/CALIB1_HRPILOT-36_BOOTH3_250US_10KHZ_9-1-26",        10000.0),
    #("Calib2 HRPILOT-36 250US",  "Calibration/HRPilot-36/CALIB2_HRPILOT-36_BOOTH2_250US_10KHZ_9-10-26",        10000.0),
    ("CM1 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM1_HRPILOT-36_BOOTH2_250US_10KHZ_9-14-26",        10000.0),
    ("CM2 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM2_HRPILOT-36_BOOTH2_250US_10KHZ_9-15-26",        10000.0),
    
    

    
    
    # Add more recordings below — uncomment or append new tuples.
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

9 recording(s) configured.
  [0] 'Calib3 HRPILOT-23 100US'  →  Calibration/HRPilot-23/CALIB3_HRPILOT-23_BOOTH3_100US_10KHZ_9-15-26  (sample_rate=10000.0 Hz)
  [1] 'Calib4 HRPILOT-23 100US'  →  Calibration/HRPilot-23/CALIB4_HRPILOT-23_BOOTH3_100US_10KHZ_9-15-26  (sample_rate=10000.0 Hz)
  [2] 'Calib4 HRPILOT-25 100US'  →  Calibration/HRPilot-25/CALIB4_HRPILOT-25_BOOTH3_100US_10KHZ_9-14-26  (sample_rate=10000.0 Hz)
  [3] 'CM1 HRPILOT-34 250US'  →  Calibration/HRPilot-34/CM1_HRPILOT-34_BOOTH1_250US_10KHZ_9-14-26  (sample_rate=10000.0 Hz)
  [4] 'CM2 HRPILOT-34 250US'  →  Calibration/HRPilot-34/CM2_HRPILOT-34_BOOTH1_250US_10KHZ_9-15-26  (sample_rate=10000.0 Hz)
  [5] 'Calib3 HRPILOT-34 100US'  →  Calibration/HRPilot-34/CALIB3_HRPILOT-34_BOOTH1_250US_10KHZ_9-15-26  (sample_rate=10000.0 Hz)
  [6] 'Calib3.5 HRPILOT-34 100US'  →  Calibration/HRPilot-34/CALIB3_HRPILOT-24_BOOTH1_250US_10KHZ_9-15-26  (sample_rate=10000.0 Hz)
  [7] 'CM1 HRPILOT-36 250US'  →  Calibration/HRPilot-36/CM1_HRPILOT-36_BO

In [38]:
# ── Load all recordings (auto-detects V2 / V3) ──────────────
# Customize FT_SNAP_HZ to match your experiment's pulse-train frequencies (Hz).
FT_SNAP_HZ_CUSTOM = FT_SNAP_HZ  # use [5.0, 10.0, 15.0, 20.0, 33.0] or override here

_all_recordings = load_all_recordings(RECORDING_DIRS, ft_snap_hz_list=FT_SNAP_HZ_CUSTOM)
_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded.  Active: {_active_rec_label!r}')


── Loading: 'Calib3 HRPILOT-23 100US'  (Calibration/HRPilot-23/CALIB3_HRPILOT-23_BOOTH3_100US_10KHZ_9-15-26)


FileNotFoundError: No .hrs* files found in 'Calibration/HRPilot-23/CALIB3_HRPILOT-23_BOOTH3_100US_10KHZ_9-15-26'

# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

`hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

In [ ]:
# Stage summary for the active recording
print(f'Active recording: {_active_rec_label!r}')
_rec_info = _all_recordings[_active_rec_label]
for _sk, (_st, _sh, _se, _slbl) in _rec_info['stage_map'].items():
    print(f'  {_slbl}: {len(_st)} trials')
print(f'\n(Run the Stage Selector cell below to enable the interactive dropdown.)')

Active recording: 'Calib4 HRPILOT-25 100US'
  MH Recruitment Curve (.hrs1): 1353 trials

(Run the Stage Selector cell below to enable the interactive dropdown.)


In [ ]:
# ── Recording & Stage Viewer Factory ───────────────────────────────────────────
# Each viewer section below has its own independent Recording + Stage dropdowns.
# Set each viewer to a different recording/stage to compare them side by side.
# make_viewer() is defined in helpers.py
from IPython.display import display as _disp

# ── Loaded recordings summary ────────────────────────────────────────────────────
print(f'{len(_all_recordings)} recording(s) loaded:')
for _rl, _rd in _all_recordings.items():
    print(f'  {_rl!r}  (App V{_rd["app_version"]}  |  {_rd["sample_rate"]} Hz)')
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        print(f'    · {_slbl}: {len(_st)} trials')
print()
print('Each viewer below has its own Recording + Stage dropdowns for independent selection.')

# ── Stimulation Intensity Histogram ─────────────────────────────────────────────────────
def _render_hist(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Histogram: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_amplitude_distribution(trials, header)

_hist_widget, _hist_render = make_viewer(_all_recordings, _active_rec_label, _render_hist)
_disp(_hist_widget)
_hist_render()

3 recording(s) loaded:
  'Calib4 HRPILOT-25 100US'  (App V2  |  10000.0 Hz)
    · MH Recruitment Curve (.hrs1): 1353 trials
  'CM1 HRPILOT-34 250US'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 1069 trials
  'CM1 HRPILOT-36 250US'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 1228 trials

Each viewer below has its own Recording + Stage dropdowns for independent selection.


In [ ]:
# ── Trial Timeline ─────────────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_tl(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Trial Timeline: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_actual_trial_timeline(trials, header=header)

_tl_widget, _tl_render = make_viewer(_all_recordings, _active_rec_label, _render_tl)
_disp(_tl_widget)
_tl_render()

# Section 3b: "Most Recent Background" + "Background EMG Level"

Recreates the H-Reflex App recruitment-curve trial-plot widgets from `MhRecruitmentCurveStage.get_trial_plot_options`:

- **Most recent background** bar chart of the pre-stim |EMG| bins.
- **EMG Level** scatter of background grand means across trials.

Bins are reconstructed from `hrs2_emg_blocks` over a fixed monitoring window (default 2500 ms ending at trigger time).

In [ ]:
# ── Background EMG Views ──────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_bg(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Background EMG: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_background_emg_views(trials, emg_blocks, monitoring_window_ms=2500)

_bg_widget, _bg_render = make_viewer(_all_recordings, _active_rec_label, _render_bg)
_disp(_bg_widget)
_bg_render()

# Section 6: HRS2 Detailed Analysis

Interactive averaged-waveform paged grid (with M/H peak markers and signal overlays) plus the normalized and raw recruitment curves. The cell below sets the analysis parameters; the cell after that calls `plot_hrs2_analysis`.

In [ ]:
#  Configuration 
PRE_PLOT_MS  = 5   # ms before stim onset to display
POST_PLOT_MS = 25  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
#M_WAVE_START_MS = 3
#M_WAVE_END_MS= 5.5
#H_WAVE_START_MS = 9.0
#H_WAVE_END_MS   = 12

M_WAVE_START_MS = 2.2 
M_WAVE_END_MS= 4.2
H_WAVE_START_MS = 6   
H_WAVE_END_MS   = 9.6

PRE_AVG_MS  = 5   # ms before stim onset
POST_AVG_MS = 25  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [ ]:
# ── Pre-compute Comparison Data ───────────────────────────────────────────────────────
# Computed once here (after configuration constants are set) for instant rendering
# in the comparison plot below. Re-run this cell if you change PRE_AVG_MS,
# POST_AVG_MS, M_WAVE_START_MS, M_WAVE_END_MS, H_WAVE_START_MS, or H_WAVE_END_MS.
_xr_cache = compute_h_comparison_data(
    _all_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)
_xr_stages = {}
for _rd in _all_recordings.values():
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        if _st and _sk not in _xr_stages:
            _xr_stages[_sk] = _slbl
print(f'Comparison data pre-computed for {len(_xr_cache)} recording(s), '
      f'{len(_xr_stages)} stage(s): {list(_xr_stages.keys())}')

Comparison data pre-computed for 3 recording(s), 2 stage(s): ['mh_recruitment', 'control_mode']


In [ ]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────────
# MERGE_ALL = True  →  collapse every amplitude into a single group.
#
# MERGED_GROUPS accepts two styles (mix freely):
#   Explicit list : [0.12, 0.13, 0.15]   — merge exactly those amplitudes
#   Range tuple   : (0.10, 0.50)         — merge all amplitudes where low <= amp <= high
#
# Examples:
#   MERGED_GROUPS = [[0.12, 0.13], [0.15, 0.16, 0.17]]   ← two explicit groups
#   MERGED_GROUPS = [(0.10, 0.20), (0.25, 0.40)]          ← two range groups
#   MERGED_GROUPS = [(0.10, 0.20), [0.50, 0.55]]          ← range + explicit, mixed
#
# Leave both flags at their defaults to use the standard (unmerged) grouping.
MERGE_ALL     = False   # True → collapse every amplitude into one group
MERGED_GROUPS = []
#[(0,0.108), (0.108,0.23), (0.23,0.43), (0.43,0.50), (0.50,0.60), (0.60,0.70)]

def _resolve_groups(trials, groups):
    _all_amps = sorted({t.stimulation_amplitude_ma for t in trials})
    resolved = []
    for g in groups:
        if isinstance(g, tuple) and len(g) == 2:
            lo, hi = float(g[0]), float(g[1])
            matched = [a for a in _all_amps if lo <= a <= hi]
            if matched:
                resolved.append(matched)
            else:
                print(f'Warning: range ({lo}, {hi}) matched no amplitudes — skipped.')
        else:
            resolved.append(list(g))
    return resolved

def _apply_merge(trials):
    if MERGE_ALL:
        _amps = sorted({t.stimulation_amplitude_ma for t in trials})
        return build_merged_amp_groups(trials, [_amps])
    if MERGED_GROUPS:
        return build_merged_amp_groups(trials, _resolve_groups(trials, MERGED_GROUPS))
    return trials

print("Merge config: MERGE_ALL =", MERGE_ALL, " | MERGED_GROUPS =", MERGED_GROUPS or "(none)")
print("Re-run viewer cells or change Recording/Stage to apply new merge settings.")

Merge config: MERGE_ALL = False  | MERGED_GROUPS = (none)
Re-run viewer cells or change Recording/Stage to apply new merge settings.


In [ ]:
# ── HRS2 Analysis: Interactive Averaged Waveforms + Recruitment Curve ─────────
from IPython.display import display as _disp

def _render_ana(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    tp = _apply_merge(trials)
    print(f'\n── Analysis: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_hrs2_analysis(
        tp, header,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
    )

_ana_widget, _ana_render = make_viewer(_all_recordings, _active_rec_label, _render_ana)
_disp(_ana_widget)
_ana_render()

# Section 6c: Frequency Test Analysis (V3 FT)

Three views available via the **View** toggle:

- **H/M MRA Per Pulse** — mean ± 1σ H-wave and M-wave MRA at each pulse position across all trials. Uses `pulse_h_wave_mra` / `pulse_m_wave_mra` pre-stored by the app (mean rectified average within each window). The most-recent trial is overlaid as a dashed line. Shows homosynaptic (rate-dependent) depression across the train.
- **Avg Waveforms** — paged 2×3 grid. Each tile is one pulse position: individual trial segments are drawn at low alpha, the bold black trace is the cross-trial mean. M-wave window is blue-shaded; H-wave window is green-shaded (matching the HRS2 Analysis viewer). MRA annotations are boxed above each window. A colorbar at the top encodes pulse # (blue = pulse 1, red = last). Use the **Page** slider to page through pulses.
- **H/M Peak Per Pulse** — same structure as H/M MRA Per Pulse, but computes `max(|EMG|)` within each wave window directly from the raw EMG trace, then averages across trials.

M/H wave windows are shared with the HRS2 configuration constants (`M_WAVE_START_MS`, `H_WAVE_START_MS`, etc.).

In [ ]:
# ── Frequency Test Analysis: Interactive Viewer ────────────────────────────────
# View toggle:  H/M MRA Per Pulse | Avg Waveforms | H/M Peak Per Pulse
# Page slider (Avg Waveforms only): step through pulse positions 6 at a time.
# M/H wave windows use M_WAVE_START_MS / H_WAVE_START_MS from the config cell above.
from ipywidgets import Dropdown, ToggleButtons, IntSlider, Output, VBox
from IPython.display import display as _disp

_ft_recs = [rl for rl in _all_recordings if _all_recordings[rl].get('ft_trials')]
if not _ft_recs:
    print("No Frequency Test data loaded (.hrft not found in any recording directory).")
else:
    _ft_rec_d = Dropdown(
        options=_ft_recs, value=_ft_recs[0],
        description='Recording:', layout={'width': '600px'}
    )
    _ft_view_d = ToggleButtons(
        options=[
            ('H/M MRA Per Pulse', 'depression'),
            ('Avg Waveforms',     'waveforms'),
            ('H/M Peak Per Pulse','peak'),
        ],
        description='View:', style={'button_width': '185px'},
    )
    # Page slider — shown only for the Avg Waveforms view
    _ft_page_s = IntSlider(
        min=1, max=1, step=1, value=1,
        description='Page:', layout={'width': '450px', 'visibility': 'hidden'}
    )
    _ft_out = Output()

    def _ft_render():
        rl   = _ft_rec_d.value
        vw   = _ft_view_d.value
        rec  = _all_recordings[rl]
        ft_t = rec.get('ft_trials', [])
        ft_h = rec.get('ft_header')
        sr   = rec.get('sample_rate') or getattr(ft_h, 'sample_rate', None)
        if not ft_t or ft_h is None:
            return

        # Update page slider max to match actual pulse count / page size
        n_p  = getattr(ft_h, 'n_pulses_per_train', 0) or \
               max((len(getattr(t, 'pulse_h_wave_mra', [])) for t in ft_t), default=1)
        tot  = max(1, int(np.ceil(n_p / N_PER_PAGE)))
        _ft_page_s.max = tot
        _ft_page_s.layout.visibility = 'visible' if vw == 'waveforms' else 'hidden'

        hz = round(1e6 / ft_h.event_period_us, 1) if getattr(ft_h, 'event_period_us', 0) else '?'
        with _ft_out:
            _ft_out.clear_output(wait=True)
            print(f'Frequency Test: {len(ft_t)} trials  |  '
                  f'{n_p} pulses/train  |  '
                  f'{hz} Hz  [{rl}]')
            if vw == 'depression':
                plot_ft_depression_curve(ft_t, ft_h, sample_rate=sr)
            elif vw == 'waveforms':
                plot_ft_averaged_waveforms(
                    ft_t, ft_h,
                    pre_pulse_ms=2.0, post_pulse_ms=POST_PLOT_MS,
                    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                    sample_rate=sr, n_per_page=N_PER_PAGE,
                    page=_ft_page_s.value - 1,
                )
            else:  # peak
                plot_ft_peak_curve(
                    ft_t, ft_h, sample_rate=sr,
                    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                )

    def _ft_on_rec(c):
        ft_t = _all_recordings[_ft_rec_d.value].get('ft_trials', [])
        _ft_page_s.value = 1
        _ft_render()

    def _ft_on_page(c):
        if _ft_view_d.value == 'waveforms':
            _ft_render()

    _ft_rec_d.observe(_ft_on_rec,                   names='value')
    _ft_view_d.observe(lambda c: _ft_render(),       names='value')
    _ft_page_s.observe(_ft_on_page,                  names='value')

    _disp(VBox([_ft_rec_d, _ft_view_d, _ft_page_s, _ft_out]))
    _ft_render()

No Frequency Test data loaded (.hrft not found in any recording directory).


# Section 6b: H-Reflex Size Across Recordings

**H-reflex size per amplitude group** is the **Mean Rectified Amplitude (MRA) of the averaged bipolar waveform** in the H-wave window, minus the MRA of the pre-stimulus background:

> **size (µV) = mean|avg_bip(t ∈ [H_START, H_END])| − mean|avg_bip(t < 0)|**

Steps per amplitude group in each recording:
1. All trials at that amplitude are time-aligned and averaged → `avg_bip`
2. **H-wave MRA** = mean of |avg_bip| within [H_WAVE_START_MS, H_WAVE_END_MS]
3. **Background MRA** = mean of |avg_bip| in the pre-stimulus window (t < 0)
4. **H-reflex size** = H-wave MRA − background MRA

This matches the green `H: X.X µV` annotation shown in the HRS2 Analysis viewer above, with background subtracted.

The plot below shows one **box-and-whisker per recording** in `RECORDING_DIRS` order, with each amplitude group contributing one data point:
- **Box** — interquartile range (25th–75th percentile across amplitude groups)
- **Horizontal bar** — median
- **◆ diamond** — mean
- **Whiskers** — mean ± 1 standard deviation
- **n=** — number of amplitude groups in that recording for the selected stage
- **Dashed line** — connects per-recording means in `RECORDING_DIRS` order

In [ ]:
# ── Cross-Recording Comparison Plot ──────────────────────────────────────────────────
# Toggle between H-Reflex size, M-Wave size, and Background MRA (pre-stim EMG level).
# Re-run the pre-compute cell above if you change analysis configuration constants.
from ipywidgets import Dropdown, ToggleButtons, Output, VBox
from IPython.display import display as _disp

_xr_stage_d = Dropdown(
    options=[(_slbl, _sk) for _sk, _slbl in _xr_stages.items()],
    description='Stage:', layout={'width': '480px'}
)
if _xr_stages:
    _xr_stage_d.value = next(iter(_xr_stages))

_xr_metric_d = ToggleButtons(
    options=[('H-Reflex Size', 'h_reflex'), ('M-Wave Size', 'm_wave'), ('Background MRA', 'background')],
    description='Metric:',
    style={'button_width': '160px'},
)

_xr_out = Output()

def _xr_render():
    sk = _xr_stage_d.value
    mt = _xr_metric_d.value
    if not sk:
        return
    with _xr_out:
        _xr_out.clear_output(wait=True)
        plot_h_reflex_comparison(_xr_cache, RECORDING_DIRS, sk, _xr_stages, metric=mt)

_xr_stage_d.observe(lambda c: _xr_render(), names='value')
_xr_metric_d.observe(lambda c: _xr_render(), names='value')
_disp(VBox([_xr_stage_d, _xr_metric_d, _xr_out]))
_xr_render()

# Section 6c: Cross-Recording DataFrame & Calculations

`xr_df` is a flat per-trial DataFrame built from `_xr_cache`.  
Re-run this cell any time you re-run the pre-compute cell above.

In [ ]:
import pandas as pd

# ── Build per-trial DataFrame from raw EMG ────────────────────────────────────
# Note: _xr_cache now stores per-amplitude-group averaged-waveform MRA (matching
# the waveform viewer). This DataFrame is independent — it computes metrics
# per individual trial directly from raw EMG for trial-level analysis.
_rows = []
for _rec_lbl, _rec in _all_recordings.items():
    _sr_xr    = _rec['sample_rate'] or SAMPLE_RATE
    _ms_ps_xr = 1000.0 / _sr_xr
    _rec_s_xr = int(TRIAL_RECORD_MS * _sr_xr / 1000)
    for _sk, (_trials, _hdr, _emg_bl, _slbl) in _rec['stage_map'].items():
        for _i, _t in enumerate(_trials):
            try:
                _tm, _et, *_ = get_trial_window(
                    _t, PRE_AVG_MS, POST_AVG_MS,
                    ms_per_sample=_ms_ps_xr, record_samples=_rec_s_xr)
                _bg_m = _tm < 0
                _bg_v = float(np.nanmean(np.abs(_et[_bg_m]))) if _bg_m.any() else 0.0
                _mm   = (_tm >= M_WAVE_START_MS) & (_tm <= M_WAVE_END_MS)
                _hm   = (_tm >= H_WAVE_START_MS) & (_tm <= H_WAVE_END_MS)
                _mv   = float(np.nanmean(np.abs(_et[_mm]))) - _bg_v if _mm.any() else float('nan')
                _hv   = float(np.nanmean(np.abs(_et[_hm]))) - _bg_v if _hm.any() else float('nan')
            except Exception:
                _mv = _hv = _bg_v = float('nan')
            _rows.append({
                'recording': _rec_lbl,
                'stage':     _slbl,
                'trial':     _i + 1,
                'h_size_uv': _hv,
                'm_size_uv': _mv,
                'bg_mra_uv': _bg_v,
                'hm_ratio':  _hv / _mv if (_mv and _mv > 0 and not np.isnan(_mv)) else float('nan'),
            })

xr_df = pd.DataFrame(_rows)
print(f"xr_df: {len(xr_df)} rows  (per-trial rectified EMG metrics)")
print(f"  recordings: {xr_df['recording'].nunique()}  |  stages: {xr_df['stage'].nunique()}")
xr_df.head(10)

xr_df: 3650 rows  (per-trial rectified EMG metrics)
  recordings: 3  |  stages: 2


,recording,stage,trial,h_size_uv,m_size_uv,bg_mra_uv,hm_ratio
0,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),1,113.426750,1389.250587,97.401634,0.081646
1,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),2,42.311035,1529.746552,117.032867,0.027659
2,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),3,102.245872,1110.754173,114.398903,0.092051
3,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),4,97.472046,850.705139,133.083618,0.114578
4,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),5,117.567863,598.991493,83.315147,0.196276
5,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),6,189.205688,836.990692,74.571136,0.226055
6,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),7,279.211868,1561.533249,80.765335,0.178806
7,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),8,77.310936,1738.321510,115.280785,0.044474
8,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),9,61.447739,1505.868591,88.090881,0.040806
9,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),10,132.352432,1499.214157,157.644608,0.088281


In [ ]:
# ── Per-recording summary statistics ─────────────────────────────────────────
xr_summary = (
    xr_df.groupby('recording')
    .agg(
        n_trials      = ('trial',      'count'),
        h_size_mean   = ('h_size_uv',  'mean'),
        h_size_std    = ('h_size_uv',  'std'),
        m_size_mean   = ('m_size_uv',  'mean'),
        m_size_std    = ('m_size_uv',  'std'),
        bg_mra_mean   = ('bg_mra_uv',  'mean'),
        hm_ratio_mean = ('hm_ratio',   'mean'),
        hm_ratio_std  = ('hm_ratio',   'std'),
    )
    .round(3)
)
xr_summary

,n_trials,h_size_mean,h_size_std,m_size_mean,m_size_std,bg_mra_mean,hm_ratio_mean,hm_ratio_std
recording,,,,,,,,
CM1 HRPILOT-34 250US,1069,279.480,228.336,1307.957,742.823,149.005,0.277,0.354
CM1 HRPILOT-36 250US,1228,655.418,226.765,1869.110,529.259,125.879,0.411,0.288
Calib4 HRPILOT-25 100US,1353,112.739,141.468,939.578,618.374,114.840,0.164,1.807


### Recording Settings DataFrame

`settings_df` captures the app-side stimulation/window settings that were active for each
(recording, stage) — parsed from the `*.settings.json` sidecar written next to each binary
data file by newer H-Reflex App recordings (sweep range, M/H-wave windows, thresholds,
polarity, pulse-train parameters, etc). Stages recorded before the app wrote sidecars still
get a row here; their settings columns are simply NaN.

In [ ]:
# ── Build settings DataFrame from *.settings.json sidecars ───────────────────────
settings_df = build_settings_dataframe(_all_recordings)
_meta_cols = {'recording', 'stage', 'stage_key', 'subject_id', 'session_start',
              'app_version', 'file_version', 'sample_rate_hz', 'n_trials'}
_setting_cols = [c for c in settings_df.columns if c not in _meta_cols]
_n_with_settings = int(settings_df[_setting_cols].notna().any(axis=1).sum()) if _setting_cols else 0
print(f"settings_df: {len(settings_df)} rows  |  {settings_df['recording'].nunique()} recording(s)  |  "
      f"{_n_with_settings}/{len(settings_df)} stage(s) have a settings.json sidecar")
settings_df.head(10)

settings_df: 3 rows  |  3 recording(s)  |  3/3 stage(s) have a settings.json sidecar


,recording,stage,stage_key,subject_id,session_start,app_version,file_version,sample_rate_hz,n_trials,user_min_stimulation_amplitude,...,manual_min_threshold,manual_max_threshold,reversed_polarity,m_wave_adjust_step_size_ma,m_wave_adjust_occurrence,m_wave_set_value_uv,m_wave_distribution_window_size,m_wave_max_intensity_ma,m_wave_min_intensity_ma,fixed_stimulation_amplitude_ma
0,Calib4 HRPILOT-25 100US,MH Recruitment Curve (.hrs1),mh_recruitment,HRPILOT-25,2026-09-14 13:37:11.744872,H-Reflex App Version 3,9,57700.0,1353,0.001,...,5.0,500.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CM1 HRPILOT-34 250US,Control Mode (.hrs2),control_mode,HRPILOT-34,2026-09-14 10:57:42.547842,H-Reflex App Version 3,10,56480.0,1069,0.135,...,25.0,315.0,True,0.001,1.0,1800.0,20.0,0.220,0.06,0.135
2,CM1 HRPILOT-36 250US,Control Mode (.hrs2),control_mode,HRPILOT-36,2026-09-14 11:01:25.298897,H-Reflex App Version 3,10,57700.0,1228,0.120,...,50.0,200.0,False,0.001,1.0,1800.0,20.0,0.165,0.08,0.120


### Save Analysis to Disk

Bundle `xr_df`, `xr_summary`, and `settings_df` into a single pickle file so this analysis
can be reloaded later without re-reading the raw binary recordings. Call
`save_analysis_pickle()` whenever you want a snapshot — nothing is saved automatically.

In [ ]:
# ── Save analysis DataFrames to a pickle file (on demand) ──────────────
import pickle
from datetime import datetime as _dt

def save_analysis_pickle(path=None, **extra_frames):
    """Pickle xr_df, xr_summary, and settings_df (plus any extra_frames) to *path*.

    path=None auto-names the file from the current timestamp in the working directory.
    Returns the path written to.
    """
    _path = path or f"h_reflex_analysis_{_dt.now():%Y%m%d_%H%M%S}.pkl"
    _bundle = {'xr_df': xr_df, 'xr_summary': xr_summary, 'settings_df': settings_df, **extra_frames}
    with open(_path, 'wb') as _f:
        pickle.dump(_bundle, _f)
    print(f"Saved {list(_bundle.keys())} -> {_path}")
    return _path

print("Ready. Call save_analysis_pickle() to pickle xr_df / xr_summary / settings_df to disk,")
print("or save_analysis_pickle('my_analysis.pkl') to choose the filename.")

Ready. Call save_analysis_pickle() to pickle xr_df / xr_summary / settings_df to disk,
or save_analysis_pickle('my_analysis.pkl') to choose the filename.


In [ ]:
# ── Custom calculations — edit this cell freely ───────────────────────────────
# Examples:

# Filter to a specific recording
# rec = xr_df[xr_df['recording'] == 'HRPILOT-23']

# Mean H-size per recording (µV)
# xr_df.groupby('recording')['h_size_uv'].mean()

# Coefficient of variation (%) for H-size per recording
# xr_df.groupby('recording')['h_size_uv'].agg(lambda x: x.std() / x.mean() * 100).rename('h_cv_pct')

# Export to CSV
# xr_df.to_csv('xr_data.csv', index=False)
# xr_summary.to_csv('xr_summary.csv')

print("xr_df and xr_summary are ready.  Edit this cell to run your calculations.")

xr_df and xr_summary are ready.  Edit this cell to run your calculations.


# Section 6d: Trial Subset Filter

Filter every recording's trials by a per-trial EMG metric before analysis.  
Set `FILTER_TARGET` to your desired M-wave (or other) size, and `FILTER_TOLERANCE_PCT` to the ± window.  
Running the filter cell creates `_filtered_recordings` — a drop-in replacement for `_all_recordings` used by the comparison plot below.  
`xr_df_filtered` is the filtered subset of the DataFrame for calculations.

In [ ]:
# ── Trial Filter Configuration ─────────────────────────────────────────────────
# Metric to filter on: 'M_WAVE' | 'H_WAVE' | 'HM_RATIO' | 'BACKGROUND'
FILTER_METRIC        = 'M_WAVE'
FILTER_TARGET        = 150.0      # µV  (or dimensionless ratio for HM_RATIO)
FILTER_TOLERANCE_PCT = 25.0       # ±%  around target

FILTER_LO = FILTER_TARGET * (1 - FILTER_TOLERANCE_PCT / 100)
FILTER_HI = FILTER_TARGET * (1 + FILTER_TOLERANCE_PCT / 100)
# Uncomment to override with manual bounds instead:
# FILTER_LO = 100.0
# FILTER_HI = 200.0

print(f"Metric  : {FILTER_METRIC}")
print(f"Target  : {FILTER_TARGET}  ±{FILTER_TOLERANCE_PCT}%")
print(f"Bounds  : [{FILTER_LO:.2f},  {FILTER_HI:.2f}]")

Metric  : M_WAVE
Target  : 150.0  ±25.0%
Bounds  : [112.50,  187.50]


In [ ]:
# ── Apply per-trial filter ─────────────────────────────────────────────────────
# Computes the chosen metric from raw EMG for every trial, keeps those in [FILTER_LO, FILTER_HI].
# Creates:
#   _filtered_recordings  — same structure as _all_recordings, with only passing trials
#   xr_df_filtered        — filtered subset of xr_df for DataFrame calculations

_filtered_recordings = {}
for _rec_lbl, _rec in _all_recordings.items():
    _sr_f    = _rec['sample_rate'] or SAMPLE_RATE
    _ms_ps_f = 1000.0 / _sr_f
    _rec_s_f = int(TRIAL_RECORD_MS * _sr_f / 1000)
    _filt_sm = {}
    for _sk, (_trials, _hdr, _emg_bl, _slbl) in _rec['stage_map'].items():
        _keep = []
        for _t in _trials:
            try:
                _tm, _et, *_ = get_trial_window(
                    _t, PRE_AVG_MS, POST_AVG_MS,
                    ms_per_sample=_ms_ps_f, record_samples=_rec_s_f,
                )
                _bg_m = _tm < 0
                _bg_v = float(np.nanmean(np.abs(_et[_bg_m]))) if _bg_m.any() else 0.0
                if FILTER_METRIC == 'M_WAVE':
                    _wm = (_tm >= M_WAVE_START_MS) & (_tm <= M_WAVE_END_MS)
                    _val = float(np.nanmean(np.abs(_et[_wm]))) - _bg_v if _wm.any() else float('nan')
                elif FILTER_METRIC == 'H_WAVE':
                    _wm = (_tm >= H_WAVE_START_MS) & (_tm <= H_WAVE_END_MS)
                    _val = float(np.nanmean(np.abs(_et[_wm]))) - _bg_v if _wm.any() else float('nan')
                elif FILTER_METRIC == 'BACKGROUND':
                    _val = _bg_v
                elif FILTER_METRIC == 'HM_RATIO':
                    _mm = (_tm >= M_WAVE_START_MS) & (_tm <= M_WAVE_END_MS)
                    _hm = (_tm >= H_WAVE_START_MS) & (_tm <= H_WAVE_END_MS)
                    _mv = float(np.nanmean(np.abs(_et[_mm]))) - _bg_v if _mm.any() else float('nan')
                    _hv = float(np.nanmean(np.abs(_et[_hm]))) - _bg_v if _hm.any() else float('nan')
                    _val = (_hv / _mv) if (_mv and not np.isnan(_mv) and _mv > 0) else float('nan')
                else:
                    _val = float('nan')
            except Exception:
                _val = float('nan')
            if not np.isnan(_val) and FILTER_LO <= _val <= FILTER_HI:
                _keep.append(_t)
        _filt_sm[_sk] = (_keep, _hdr, _emg_bl, _slbl)
    _filtered_recordings[_rec_lbl] = {**_rec, 'stage_map': _filt_sm}

# ── Filtered DataFrame (from xr_df by metric column) ──────────────────────────
_filt_col = {
    'M_WAVE': 'm_size_uv', 'H_WAVE': 'h_size_uv',
    'HM_RATIO': 'hm_ratio', 'BACKGROUND': 'bg_mra_uv',
}.get(FILTER_METRIC, 'm_size_uv')
xr_df_filtered = xr_df[
    (xr_df[_filt_col] >= FILTER_LO) & (xr_df[_filt_col] <= FILTER_HI)
].copy().reset_index(drop=True)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"Filter: {FILTER_METRIC}  in  [{FILTER_LO:.2f}, {FILTER_HI:.2f}]\n")
_tot_orig = _tot_kept = 0
for _rl, _fr in _filtered_recordings.items():
    for _sk, (_ft, *_) in _fr['stage_map'].items():
        _orig = len(_all_recordings[_rl]['stage_map'][_sk][0])
        _kept = len(_ft)
        _tot_orig += _orig
        _tot_kept += _kept
        _pct = 100 * _kept / _orig if _orig else 0
        print(f"  {_rl}  [{_all_recordings[_rl]['stage_map'][_sk][3]}]:"
              f"  {_kept}/{_orig} trials  ({_pct:.1f}%)")
if _tot_orig:
    print(f"\n  Total: {_tot_kept}/{_tot_orig} trials  ({100*_tot_kept/_tot_orig:.1f}%)")
print(f"\nxr_df_filtered: {len(xr_df_filtered)} / {len(xr_df)} rows")

Filter: M_WAVE  in  [112.50, 187.50]

  Calib4 HRPILOT-25 100US  [MH Recruitment Curve (.hrs1)]:  48/1353 trials  (3.5%)
  CM1 HRPILOT-34 250US  [Control Mode (.hrs2)]:  4/1069 trials  (0.4%)
  CM1 HRPILOT-36 250US  [Control Mode (.hrs2)]:  0/1228 trials  (0.0%)

  Total: 52/3650 trials  (1.4%)

xr_df_filtered: 52 / 3650 rows


In [ ]:
# ── Filtered Comparison Plot ───────────────────────────────────────────────────
# Re-computes the cross-recording cache on filtered trials and shows the comparison.
# Re-run this cell any time you change the filter bounds above.
from ipywidgets import Dropdown, ToggleButtons, Output, VBox
from IPython.display import display as _disp

_xr_cache_filt = compute_h_comparison_data(
    _filtered_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)

_xr_filt_out = Output()

def _xr_filt_render():
    _sk = _xr_filt_stage_d.value
    _mt = _xr_filt_metric_d.value
    if not _sk:
        return
    with _xr_filt_out:
        _xr_filt_out.clear_output(wait=True)
        plot_h_reflex_comparison(_xr_cache_filt, RECORDING_DIRS, _sk, _xr_stages, metric=_mt)

_xr_filt_stage_d = Dropdown(
    options=[(_slbl, _sk) for _sk, _slbl in _xr_stages.items()],
    description='Stage:', layout={'width': '480px'},
)
if _xr_stages:
    _xr_filt_stage_d.value = next(iter(_xr_stages))

_xr_filt_metric_d = ToggleButtons(
    options=[('H-Reflex Size', 'h_reflex'), ('M-Wave Size', 'm_wave'), ('Background MRA', 'background')],
    description='Metric:', style={'button_width': '160px'},
)

_xr_filt_stage_d.observe(lambda c: _xr_filt_render(), names='value')
_xr_filt_metric_d.observe(lambda c: _xr_filt_render(), names='value')
_disp(VBox([_xr_filt_stage_d, _xr_filt_metric_d, _xr_filt_out]))
_xr_filt_render()

# Section 3c: H:M Ratio Summary

Box plot and histogram of H:M ratio for each stimulation polarity group.
If both normal and reversed polarities were used in this session, each group is analysed separately.

In [ ]:
# ── Stim Polarity Analysis ────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_pol(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Polarity / H:M Ratio: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    trials_by_polarity = split_trials_by_polarity(trials)
    plot_hm_ratio_summary(
        trials_by_polarity, header,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
    )
    plot_hwave_regression(
        trials, emg_blocks,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
    )

_pol_widget, _pol_render = make_viewer(_all_recordings, _active_rec_label, _render_pol)
_disp(_pol_widget)
_pol_render()

# Section 3d: M-Wave Stabilization Control Error (V3 S2 / S4 / S5 / S6)

Plots the M-wave stabilization controller output trial-by-trial.
- **Left axis** — `m_wave_error` (µV): difference between actual M-wave and the target set-point; falls back to `m_wave_window_median` when the controller was inactive.
- **Right axis** — `stimulation_amplitude_ma` (mA, orange): stimulation intensity actually used.

Available only for V3 recordings (file_version ≥ 9 for S2; file_version ≥ 2 for S4/S5/S6).

In [ ]:
# ── M-Wave Control Error with Reference Lines ─────────────────────────────────
# Controls:
#   Auto checkbox  — reads target M-size from the recording's stored set-value
#   Target (µV)    — manual override (editable when Auto is unchecked)
#   ±Inner %       — tolerance band drawn as green dashed lines around the target
#   ±Outer %       — algorithm bounds drawn as red dotted lines (set to 0 to hide)
#   Update Plot    — re-render with current settings after changing parameters
from ipywidgets import Checkbox, FloatText, FloatSlider, Button, Output, VBox, HBox, Label
from IPython.display import display as _disp
import numpy as np

def _mw_stage_filter(key, trials, header, emg_blocks, label):
    return bool(trials) and any(
        not np.isnan(getattr(t, 'm_wave_error', float('nan'))) or
        not np.isnan(getattr(t, 'm_wave_window_median', float('nan')))
        for t in trials)

def _mw_auto_target(trials):
    vals  = [getattr(t, 'm_wave_set_value_uv', float('nan')) for t in trials]
    valid = [v for v in vals if not np.isnan(v)]
    return float(np.median(valid)) if valid else float('nan')

# ── Control widgets ────────────────────────────────────────────────────────────
_mw_auto_cb   = Checkbox(value=True, description='Auto (recording set-value)',
                          style={'description_width': 'initial'}, layout={'width': '270px'})
_mw_target_ft = FloatText(value=0.0, description='Target (µV):',
                           step=10, disabled=True, layout={'width': '190px'})
_mw_inner_sl  = FloatSlider(value=25.0, min=0, max=100, step=1,
                              description='±Inner %:', readout_format='.0f',
                              style={'description_width': '70px'}, layout={'width': '280px'})
_mw_outer_sl  = FloatSlider(value=50.0, min=0, max=200, step=1,
                              description='±Outer %:', readout_format='.0f',
                              style={'description_width': '70px'}, layout={'width': '280px'})
_mw_apply_btn = Button(description='Update Plot', button_style='primary',
                        layout={'width': '120px'})
_mw_info_lbl  = Label(value='')

_mw_controls = VBox([
    HBox([_mw_auto_cb, _mw_target_ft, _mw_info_lbl]),
    HBox([_mw_inner_sl, _mw_outer_sl, _mw_apply_btn]),
])

def _render_mw(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    if _mw_auto_cb.value:
        _auto = _mw_auto_target(trials)
        if not np.isnan(_auto):
            _mw_target_ft.value = _auto
            _mw_info_lbl.value  = f'Auto: {_auto:.1f} µV'
        else:
            _mw_info_lbl.value  = 'Auto: no set-value stored'
        _target = None
    else:
        _target = _mw_target_ft.value
        _mw_info_lbl.value = f'Override: {_target:.1f} µV'

    print(f'\n── M-Wave Control Error: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_mwave_control_error(
        trials, header,
        sample_rate=sr or h1h.sample_rate,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
        target_uv=_target,
        inner_pct=_mw_inner_sl.value,
        outer_pct=_mw_outer_sl.value,
    )

_has_any_mw = any(
    any(_mw_stage_filter(sk, _t, _h, _e, lbl)
        for sk, (_t, _h, _e, lbl) in rd['stage_map'].items())
    for rd in _all_recordings.values()
)

if _has_any_mw:
    _mw_widget, _mw_render = make_viewer(
        _all_recordings, _active_rec_label, _render_mw,
        stage_filter=_mw_stage_filter,
    )

    def _mw_auto_toggle(change):
        _mw_target_ft.disabled = change['new']
        _mw_render()

    _mw_auto_cb.observe(_mw_auto_toggle, names='value')
    _mw_apply_btn.on_click(lambda b: _mw_render())

    _disp(_mw_controls)
    _disp(_mw_widget)
    _mw_render()
else:
    print("No M-wave stabilization data available in any loaded recording "
          "(requires V3 S2 file_version ≥ 9, or S4/S5/S6 file_version ≥ 2).")

In [ ]:
# ── HRS2 Trial Viewer: Interactive Per-Trial Grid + Zoom ──────────────────────
from IPython.display import display as _disp

def _render_trv(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    tp = _apply_merge(trials)
    print(f'\n── Trial Viewer: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_hrs2_trials(
        tp, header,
        pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
    )

_trv_widget, _trv_render = make_viewer(_all_recordings, _active_rec_label, _render_trv)
_disp(_trv_widget)
_trv_render()